# Understanding IFC Data for openBIM Workflows and Tools – Notebook

This notebook is the practical part of the tutorial. It goes through the basic IfcOpenShell commands, that can be used to explore what data is accessible in an IFC model. The commands are the same as shown in the table in the README.

The notebook can be run on any of the course models. Comprehending what data is actually available in the model, should be the first step when outlining what tools are possible to build.

## 1. Importing the model

First IfcOpenShell is imported. The `ifcopenshell.util.element` module is imported as `Element`, since most of the commands used to extract data are found here. Remember to change the path to the model you want to explore.

If IfcOpenShell is not installed, run `pip install ifcopenshell` first.

In [ ]:
import ifcopenshell
import ifcopenshell.util.element as Element

model = ifcopenshell.open("PATH_TO_MODEL.ifc")

## 2. Schema of the model

The IFC file contains information on what IFC schema (version) the model is based on. This is relevant to know, since some classes and attributes differ between fx IFC2X3 and IFC4.

In [ ]:
model.schema

## 3. Selecting a class

`model.by_type()` selects the whole class of a given type. Fx using `"IfcWall"` will return all walls in the model. Since IFC is built as a hierarchy, it is also possible to use a more abstract class such as `"IfcBuildingElement"`, which will return all walls, beams, slabs etc.

In [ ]:
walls = model.by_type("IfcWall")
print("Number of walls:", len(walls))

for wall in walls[:5]:
    print(wall.is_a(), wall.Name)

## 4. Direct attributes of an object

`get_info()` returns the direct attributes of an object, such as `GlobalId`, `Name`, `ObjectType` and `PredefinedType`. These are the attributes defined by the class itself, and not the properties added in property sets.

In [ ]:
wall = walls[0]
wall.get_info()

## 5. Property and quantity sets

`Element.get_psets()` returns all property and quantity sets assigned to an object, including the sets inherited from the object type. The result is a dictionary, where each set contains its properties or quantities.

It is also possible to only receive the property sets or the quantity sets by using `psets_only=True` or `qtos_only=True`.

In [ ]:
psets = Element.get_psets(wall)

for set_name, properties in psets.items():
    print(set_name)
    for name, value in properties.items():
        if name != "id":
            print("   ", name, ":", value)

In [ ]:
# Only the quantity sets
Element.get_psets(wall, qtos_only=True)

Note that if the model has not been exported with base quantities, there will be no quantity sets (`Qto_...`). This does not mean the code is wrong, but that the data is not available in the model.

## 6. Material

`Element.get_material()` returns the material of an object. For objects with several layers, fx a wall, it will often return a layer set containing each material layer.

In [ ]:
material = Element.get_material(wall)
print(material)

## 7. Object type and occurrences

`Element.get_type()` returns the object type of an object, fx a specific `IfcWallType`. The other way around, `Element.get_types()` returns all objects (occurrences) of a given type. This makes it possible to find all walls sharing the same characteristics.

In [ ]:
wall_type = Element.get_type(wall)
print("Type:", wall_type)

if wall_type:
    occurrences = Element.get_types(wall_type)
    print("Number of occurrences of this type:", len(occurrences))

## 8. Spatial container

`Element.get_container()` returns the spatial container of an object, fx the storey the object is placed on. This is useful when wanting to sort or summarise objects per storey.

In [ ]:
storey = Element.get_container(wall)
print(wall.Name, "is placed on", storey.Name if storey else None)

## 9. Combining the commands

The commands can be combined to get an overview of what data is available for a whole class. The example below finds all property and quantity sets used by the walls in the model, and which properties they contain. This is a good starting point, when outlining the use case of a tool.

In [ ]:
from collections import defaultdict

overview = defaultdict(set)
for wall in model.by_type("IfcWall"):
    for set_name, properties in Element.get_psets(wall).items():
        overview[set_name].update(name for name in properties if name != "id")

for set_name, names in sorted(overview.items()):
    print(set_name, "->", sorted(names))